# Notebook 4 — Dashboard & End-to-End Demo

**What you'll do:**
1. Generate sample review data
2. Visualise score trends and severity breakdowns with Plotly
3. Run an end-to-end manual review via the API
4. Checklist for pushing to GitHub

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
print('Imports OK.')

## Simulate Review History

In [ ]:
import random
random.seed(42)

sample_reviews = [
    {
        'pr_number': i + 1,
        'pr_title': random.choice([
            'Add user authentication', 'Fix N+1 query in dashboard',
            'Refactor payment module', 'Add input validation', 'Update API endpoints',
            'Fix XSS vulnerability', 'Optimise database indexes', 'Add rate limiting',
        ]),
        'score': random.randint(3, 10),
        'issues': random.randint(0, 6),
        'severities': random.choices(['critical','high','medium','low'], k=random.randint(0,5)),
    }
    for i in range(15)
]

df = pd.DataFrame(sample_reviews)
df.head(10)

## Score Trend Over PRs

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df['pr_number'], y=df['score'],
    mode='lines+markers',
    line=dict(color='#6366f1', width=2),
    marker=dict(size=8, color=df['score'],
                colorscale=[[0,'#f87171'],[0.5,'#fbbf24'],[1,'#34d399']],
                showscale=True, cmin=1, cmax=10),
    name='Score',
))
fig.add_hline(y=7, line_dash='dash', line_color='#34d399', annotation_text='Good threshold (7)')
fig.update_layout(
    title='Code Quality Score per Pull Request',
    xaxis_title='PR Number', yaxis_title='Score (1–10)',
    yaxis=dict(range=[0, 11]),
    template='plotly_dark', height=350,
)
fig.show()

## Severity Breakdown

In [ ]:
from collections import Counter
all_severities = [s for r in sample_reviews for s in r['severities']]
counts = Counter(all_severities)

ORDER = ['critical', 'high', 'medium', 'low']
colors = {'critical': '#f87171', 'high': '#fb923c', 'medium': '#fbbf24', 'low': '#60a5fa'}

fig2 = go.Figure(go.Bar(
    x=[counts.get(s, 0) for s in ORDER],
    y=ORDER,
    orientation='h',
    marker_color=[colors[s] for s in ORDER],
))
fig2.update_layout(
    title='Total Issues by Severity',
    xaxis_title='Count', template='plotly_dark', height=280,
)
fig2.show()

## Score Distribution Histogram

In [ ]:
fig3 = px.histogram(
    df, x='score', nbins=10,
    color_discrete_sequence=['#6366f1'],
    title='Score Distribution Across All PRs',
    template='plotly_dark',
    labels={'score': 'Score (1–10)'},
)
fig3.update_layout(height=300)
fig3.show()

print(f"Average score : {df['score'].mean():.1f}")
print(f"PRs needing attention (score < 5): {len(df[df['score'] < 5])}")

## End-to-End: Manual Review via API

Make sure the Flask server is running (`python -m flask --app app.webhook_handler run --port 5001`), then run this cell.

In [ ]:
import requests, json
from IPython.display import Markdown, display

test_diff = '''
### File: utils.py (added)
@@ -0,0 +1,10 @@
+import pickle
+
+def load_user_data(raw_bytes):
+    # Deserialising untrusted data — arbitrary code execution risk!
+    return pickle.loads(raw_bytes)
+
+def get_users():
+    password = "admin123"  # hardcoded credential
+    return fetch_from_db(password)
'''

try:
    resp = requests.post(
        'http://localhost:5001/api/review',
        json={'diff': test_diff, 'pr_title': 'Add user utility functions'},
        timeout=30,
    )
    result = resp.json()
    display(Markdown(result.get('markdown', 'No markdown returned.')))
except requests.ConnectionError:
    print('Server not running — start it first.')

## Checklist — Push to GitHub

Run this cell to verify your project is ready to push.

In [ ]:
import os, pathlib

ROOT = pathlib.Path('..').resolve()

checks = [
    ('requirements.txt exists',    (ROOT / 'requirements.txt').exists()),
    ('.env.example exists',        (ROOT / '.env.example').exists()),
    ('.env NOT committed',          not (ROOT / '.env').exists() or '.env' in open(ROOT / '.gitignore').read() if (ROOT / '.gitignore').exists() else True),
    ('README.md exists',           (ROOT / 'README.md').exists()),
    ('app/webhook_handler.py',     (ROOT / 'app/webhook_handler.py').exists()),
    ('app/ai_reviewer.py',         (ROOT / 'app/ai_reviewer.py').exists()),
    ('app/github_client.py',       (ROOT / 'app/github_client.py').exists()),
    ('tests/ exists',              (ROOT / 'tests').exists()),
    ('notebooks/ exists',          (ROOT / 'notebooks').exists()),
]

all_ok = True
for name, ok in checks:
    status = '✅' if ok else '❌'
    print(f'{status}  {name}')
    if not ok:
        all_ok = False

print()
if all_ok:
    print('🎉 All checks passed — ready to push!')
    print()
    print('Commands to push to GitHub:')
    print('  git init')
    print('  git add .')
    print('  git commit -m "Initial commit: AI Code Review Assistant"')
    print('  git remote add origin https://github.com/<your-username>/ai-code-reviewer.git')
    print('  git push -u origin main')
else:
    print('⚠️  Some checks failed — see above.')